In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
%pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 72.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

import dagshub
dagshub.init(repo_owner='amama22', repo_name='ieee-cis-fraud-detection', mlflow=True)
mlflow.set_experiment("LightGBM_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2868d732-7287-47fa-a35f-429d84087926&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a977eb04652dccefaf1989bfbdfd0cb2ec3477e4b38e7eaa5994586b4c9b8898




Accessing as amama22

Initialized MLflow to track repo "amama22/ieee-cis-fraud-detection"

Repository amama22/ieee-cis-fraud-detection initialized!

<Experiment: artifact_location='mlflow-artifacts:/b8a936add5044749aaeb2584ed7362f0', creation_time=1778106515815, experiment_id='1', last_update_time=1778106515815, lifecycle_stage='active', name='LightGBM_Training', tags={}, trace_location=None, workspace='default'>

In [6]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
print(train[['TransactionDT']].describe())
print("\nD columns sample:\n", train[['D1','D2','D3','D4','D5','D10','D15']].describe())

       TransactionDT
count   5.905400e+05
mean    7.372311e+06
std     4.617224e+06
min     8.640000e+04
25%     3.027058e+06
50%     7.306528e+06
75%     1.124662e+07
max     1.581113e+07

D columns sample:
                   D1             D2             D3             D4  \
count  589271.000000  309743.000000  327662.000000  421618.000000   
mean       94.347568     169.563231      28.343348     140.002441   
std       157.660387     177.315865      62.384721     191.096774   
min         0.000000       0.000000       0.000000    -122.000000   
25%         0.000000      26.000000       1.000000       0.000000   
50%         3.000000      97.000000       8.000000      26.000000   
75%       122.000000     276.000000      27.000000     253.000000   
max       640.000000     640.000000     819.000000     869.000000   

                  D5            D10            D15  
count  280699.000000  514518.000000  501427.000000  
mean       42.335965     123.982137     163.744579  
std       

In [7]:
print("D1 fraud correlation:")
print(train.groupby('isFraud')['D1'].describe())

D1 fraud correlation:
            count       mean         std  min  25%  50%    75%    max
isFraud                                                              
0        568654.0  96.364705  158.973258  0.0  0.0  4.0  126.0  640.0
1         20617.0  38.711306  100.915599  0.0  0.0  0.0   14.0  637.0


In [8]:
print("Hour of day sample:")
print((train['TransactionDT'] % 86400 / 3600).describe())

print("\nDay of week sample:")
print(((train['TransactionDT'] // 86400) % 7).value_counts().sort_index())

print("\nFraud rate by D1 bucket:")
train['D1_bucket'] = pd.cut(train['D1'], bins=[-1, 0, 7, 30, 100, 640])
print(train.groupby('D1_bucket', observed=True)['isFraud'].mean())

Hour of day sample:
count    590540.000000
mean         14.363484
std           7.618839
min           0.000000
25%           6.785625
50%          16.840000
75%          20.408889
max          23.999722
Name: TransactionDT, dtype: float64

Day of week sample:
TransactionDT
0    86377
1    98502
2    79834
3    70223
4    85433
5    84815
6    85356
Name: count, dtype: int64

Fraud rate by D1 bucket:
D1_bucket
(-1, 0]       0.042084
(0, 7]        0.093894
(7, 30]       0.041808
(30, 100]     0.022439
(100, 640]    0.015034
Name: isFraud, dtype: float64


In [9]:
train['hour'] = (train['TransactionDT'] % 86400 / 3600).astype(int)
print(train.groupby('hour')['isFraud'].mean().sort_values(ascending=False).head(10))

hour
7     0.106102
8     0.093014
9     0.089956
6     0.077743
5     0.070302
10    0.053212
4     0.051890
11    0.038816
3     0.038314
2     0.037483
Name: isFraud, dtype: float64


In [7]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
train = train.merge(train_id, on='TransactionID', how='left')
test = test.merge(test_id, on='TransactionID', how='left')

TARGET = 'isFraud'
X = train.drop(columns=[TARGET, 'TransactionID'])
y = train[TARGET]
X_test = test.drop(columns=['TransactionID'])

print("Train shape after merge:", X.shape)
print("Test shape after merge:", X_test.shape)

Train shape after merge: (590540, 432)
Test shape after merge: (506691, 432)


# cleaning

In [8]:
class LGBMCleaningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, null_threshold=0.8):
        self.null_threshold = null_threshold
    
    def fit(self, X, y=None):
        null_pct = X.isnull().sum() / len(X)
        self.high_null_cols_ = null_pct[null_pct > self.null_threshold].index.tolist()
        self.always_drop_ = [c for c in ['M1'] if c in X.columns]
        self.cols_to_drop_ = list(set(self.high_null_cols_ + self.always_drop_))
        
        if 'card6' in X.columns:
            card6_counts = X['card6'].value_counts()
            self.rare_card6_ = card6_counts[card6_counts < 100].index.tolist()
        else:
            self.rare_card6_ = []
        
        self.d_cols_ = [c for c in X.columns 
                        if c.startswith('D') and c not in self.cols_to_drop_]
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])
        
        if 'card6' in X.columns:
            X['card6'] = X['card6'].apply(
                lambda x: 'other' if x in self.rare_card6_ else x
            )
        
        for col in self.d_cols_:
            if col in X.columns and pd.api.types.is_numeric_dtype(X[col]):
                X[col] = X[col].clip(lower=0)
        
        return X

In [9]:
X = X.drop(columns=['hour', 'D1_bucket'], errors='ignore')
X_test = X_test.drop(columns=['hour', 'D1_bucket'], errors='ignore')
print("Cleaned X shape:", X.shape)

Cleaned X shape: (590540, 432)


In [21]:
mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_Cleaning_V1"):
    cleaner_v1 = LGBMCleaningTransformer(null_threshold=0.80)
    cleaner_v1.fit(X)
    X_v1 = cleaner_v1.transform(X)
    mlflow.log_param("null_threshold", 0.80)
    mlflow.log_param("clips_negative_D", True)
    mlflow.log_metric("cols_dropped", len(cleaner_v1.cols_to_drop_))
    mlflow.log_metric("remaining_features", X_v1.shape[1])
    print(f"V1: {X.shape[1]} → {X_v1.shape[1]}")

with mlflow.start_run(run_name="LightGBM_Cleaning_V2"):
    cleaner_v2 = LGBMCleaningTransformer(null_threshold=0.50)
    cleaner_v2.fit(X)
    X_v2 = cleaner_v2.transform(X)
    mlflow.log_param("null_threshold", 0.50)
    mlflow.log_param("clips_negative_D", True)
    mlflow.log_metric("cols_dropped", len(cleaner_v2.cols_to_drop_))
    mlflow.log_metric("remaining_features", X_v2.shape[1])
    print(f"V2: {X.shape[1]} → {X_v2.shape[1]}")

with mlflow.start_run(run_name="LightGBM_Cleaning_V3"):
    cleaner_v3 = LGBMCleaningTransformer(null_threshold=0.90)
    cleaner_v3.fit(X)
    X_v3 = cleaner_v3.transform(X)
    mlflow.log_param("null_threshold", 0.90)
    mlflow.log_param("clips_negative_D", True)
    mlflow.log_metric("cols_dropped", len(cleaner_v3.cols_to_drop_))
    mlflow.log_metric("remaining_features", X_v3.shape[1])
    print(f"V3: {X.shape[1]} → {X_v3.shape[1]}")

V1: 432 → 357
🏃 View run LightGBM_Cleaning_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/8c659dd646ce417f9cdd26612bad4da1
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
V2: 432 → 217
🏃 View run LightGBM_Cleaning_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/0630eece428f4438ae7c0f0dd93f8fbb
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
V3: 432 → 419
🏃 View run LightGBM_Cleaning_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/36abbd66839c48f39b2f14b1fe616fd2
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1


# feature engineering

In [10]:
class LGBMFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, version=1):
        self.version = version
    
    def fit(self, X, y=None):
        if self.version >= 2 and 'P_emaildomain' in X.columns:
            self.email_freq_ = X['P_emaildomain'].value_counts(normalize=True).to_dict()
        if self.version == 3 and 'card1' in X.columns:
            self.card1_amt_mean_ = X.groupby('card1')['TransactionAmt'].mean().to_dict()
            self.global_amt_mean_ = X['TransactionAmt'].mean()
        return self
    
    def transform(self, X):
        X = X.copy()
        
        if 'TransactionAmt' in X.columns:
            X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        
        if 'id_01' in X.columns:
            X['has_identity'] = X['id_01'].notna().astype(int)
        
        if 'TransactionDT' in X.columns:
            X['transaction_hour'] = (X['TransactionDT'] % 86400 / 3600).astype(int)
            X['transaction_dow'] = (X['TransactionDT'] // 86400 % 7).astype(int)
            X['is_high_fraud_hour'] = X['transaction_hour'].between(5, 10).astype(int)
        
        if self.version >= 2:
            if 'P_emaildomain' in X.columns:
                high_risk = ['protonmail.com', 'mail.com', 'outlook.es', 'aim.com']
                mid_risk  = ['outlook.com', 'hotmail.es', 'live.com.mx', 'hotmail.com']
                X['email_risk_tier'] = X['P_emaildomain'].apply(
                    lambda x: 2 if x in high_risk else (1 if x in mid_risk else 0)
                )
                X['P_emaildomain_freq'] = (X['P_emaildomain']
                                           .map(self.email_freq_).fillna(0))
            
            if 'D1' in X.columns:
                X['D1_is_new_account'] = (X['D1'] <= 7).astype(int)
        
        if self.version == 3:
            if 'transaction_hour' in X.columns:
                X['hour_sin'] = np.sin(2 * np.pi * X['transaction_hour'] / 24)
                X['hour_cos'] = np.cos(2 * np.pi * X['transaction_hour'] / 24)
            
            if 'D1' in X.columns and 'D10' in X.columns:
                X['D1_D10_ratio'] = X['D1'] / (X['D10'] + 1)
            
            if 'card1' in X.columns and 'TransactionAmt' in X.columns:
                X['amt_vs_card_mean'] = (
                    X['TransactionAmt'] /
                    X['card1'].map(self.card1_amt_mean_)
                    .fillna(self.global_amt_mean_)
                )
        
        return X

In [25]:
with mlflow.start_run(run_name="LightGBM_Feature_Engineering_V1"):
    fe_v1 = LGBMFeatureEngineer(version=1)
    fe_v1.fit(X_v1)
    X_fe_v1 = fe_v1.transform(X_v1)
    mlflow.log_param("version", 1)
    mlflow.log_param("new_features", ['log_TransactionAmt', 'has_identity',
                                      'transaction_hour', 'transaction_dow',
                                      'is_high_fraud_hour'])
    mlflow.log_metric("features_after_engineering", X_fe_v1.shape[1])
    print(f"V1: {X_v1.shape[1]} → {X_fe_v1.shape[1]}")

with mlflow.start_run(run_name="LightGBM_Feature_Engineering_V2"):
    fe_v2 = LGBMFeatureEngineer(version=2)
    fe_v2.fit(X_v2)
    X_fe_v2 = fe_v2.transform(X_v2)
    mlflow.log_param("version", 2)
    mlflow.log_param("new_features", ['log_TransactionAmt', 'has_identity',
                                      'transaction_hour', 'transaction_dow',
                                      'is_high_fraud_hour', 'email_risk_tier',
                                      'P_emaildomain_freq', 'D1_is_new_account'])
    mlflow.log_metric("features_after_engineering", X_fe_v2.shape[1])
    print(f"V2: {X_v2.shape[1]} → {X_fe_v2.shape[1]}")

with mlflow.start_run(run_name="LightGBM_Feature_Engineering_V3"):
    fe_v3 = LGBMFeatureEngineer(version=3)
    fe_v3.fit(X_v3)
    X_fe_v3 = fe_v3.transform(X_v3)
    mlflow.log_param("version", 3)
    mlflow.log_param("new_features", ['log_TransactionAmt', 'has_identity',
                                      'transaction_hour', 'transaction_dow',
                                      'is_high_fraud_hour', 'email_risk_tier',
                                      'P_emaildomain_freq', 'D1_is_new_account',
                                      'hour_sin', 'hour_cos', 'D1_D10_ratio',
                                      'amt_vs_card_mean'])
    mlflow.log_metric("features_after_engineering", X_fe_v3.shape[1])
    print(f"V3: {X_v3.shape[1]} → {X_fe_v3.shape[1]}")

V1: 357 → 362
🏃 View run LightGBM_Feature_Engineering_V1 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/806c2eb50e9a4a86bab74a4e42780756
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
V2: 217 → 224
🏃 View run LightGBM_Feature_Engineering_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/798a6d1a6387408eb9b024364d27eb23
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
V3: 419 → 431
🏃 View run LightGBM_Feature_Engineering_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/cdda74bf41ee49ebbe10e6cfd565757b
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1


In [11]:
class LGBMCategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, use_native_categoricals=False):
        self.use_native_categoricals = use_native_categoricals
    
    def fit(self, X, y=None):
        self.m_cols_ = [c for c in X.columns
                        if c.startswith('M') and X[c].dtype == 'object']
        self.other_cat_cols_ = [c for c in X.columns
                                if X[c].dtype == 'object'
                                and c not in self.m_cols_]
        if not self.use_native_categoricals:
            self.ordinal_enc_ = OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )
            if self.other_cat_cols_:
                self.ordinal_enc_.fit(X[self.other_cat_cols_].astype(str))
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.m_cols_:
            if col in X.columns:
                X[col] = X[col].map({'T': 1, 'F': 0}).fillna(-1)
        
        if self.other_cat_cols_:
            for col in self.other_cat_cols_:
                if col not in X.columns:
                    X[col] = np.nan
            
            X[self.other_cat_cols_] = self.ordinal_enc_.transform(
                X[self.other_cat_cols_].astype(str)
            )
        
        return X

In [11]:
enc_v1 = LGBMCategoricalEncoder(use_native_categoricals=False)
enc_v1.fit(X_fe_v1)
X_enc_v1 = enc_v1.transform(X_fe_v1)

enc_v2 = LGBMCategoricalEncoder(use_native_categoricals=True)
enc_v2.fit(X_fe_v2)
X_enc_v2 = enc_v2.transform(X_fe_v2)

enc_v3 = LGBMCategoricalEncoder(use_native_categoricals=True)
enc_v3.fit(X_fe_v3)
X_enc_v3 = enc_v3.transform(X_fe_v3)

print("V1 object cols remaining:", X_enc_v1.select_dtypes('object').columns.tolist())
print("V2 category cols:", X_enc_v2.select_dtypes('category').columns.tolist())
print("V3 category cols:", X_enc_v3.select_dtypes('category').columns.tolist())
print(f"\nShapes: V1={X_enc_v1.shape}, V2={X_enc_v2.shape}, V3={X_enc_v3.shape}")

NameError: name 'X_fe_v1' is not defined

# feature selection

In [12]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold

class VarianceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        self.selector_ = VarianceThreshold(threshold=self.threshold)
        self.selector_.fit(X.select_dtypes(exclude='category'))
        numeric_cols = X.select_dtypes(exclude='category').columns
        self.selected_cols_ = (
            numeric_cols[self.selector_.get_support()].tolist() +
            X.select_dtypes(include='category').columns.tolist()
        )
        self.dropped_cols_ = numeric_cols[~self.selector_.get_support()].tolist()
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]


class CorrelationSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
    
    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(exclude='category').columns
        corr_matrix = X[numeric_cols].corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.dropped_cols_ = [col for col in upper.columns
                              if any(upper[col] > self.threshold)]
        self.selected_cols_ = (
            [c for c in numeric_cols if c not in self.dropped_cols_] +
            X.select_dtypes(include='category').columns.tolist()
        )
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]

class LGBMImportanceSelector(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=150):
        self.top_n = top_n
    
    def fit(self, X, y=None):
        sample_size = min(50000, len(X))
        idx = np.random.choice(len(X), size=sample_size, replace=False)
        X_sample = X.iloc[idx]
        y_sample = y.iloc[idx]
        
        quick_model = LGBMClassifier(
            n_estimators=50,
            max_depth=4,
            learning_rate=0.1,
            random_state=42,
            verbose=-1
        )
        quick_model.fit(X_sample, y_sample)
        
        importances = pd.Series(
            quick_model.feature_importances_,
            index=X.columns
        ).sort_values(ascending=False)
        
        self.selected_cols_ = importances.head(self.top_n).index.tolist()
        self.dropped_cols_ = importances.tail(len(X.columns) - self.top_n).index.tolist()
        self.importances_ = importances
        return self
    
    def transform(self, X):
        return X[self.selected_cols_]

# training

In [19]:
def process_variation(X, y, version, null_threshold, use_native_cat,
                       selector_class, selector_params, run_prefix, log=True):
    from contextlib import nullcontext
    def maybe_run(name):
        return mlflow.start_run(run_name=name) if log else nullcontext()
    
    
    with maybe_run(f"LightGBM_Cleaning_{run_prefix}"):
        cleaner = LGBMCleaningTransformer(null_threshold=null_threshold)
        cleaner.fit(X)
        X_clean = cleaner.transform(X)
        mlflow.log_param("null_threshold", null_threshold)
        mlflow.log_metric("cols_dropped", len(cleaner.cols_to_drop_))
        mlflow.log_metric("remaining_features", X_clean.shape[1])

    with maybe_run(f"LightGBM_Feature_Engineering_{run_prefix}"):
        fe = LGBMFeatureEngineer(version=version)
        fe.fit(X_clean)
        X_fe = fe.transform(X_clean)
        del X_clean; gc.collect()
        mlflow.log_param("version", version)
        mlflow.log_metric("features_after_engineering", X_fe.shape[1])

    with maybe_run(f"LightGBM_Encoding_{run_prefix}"):
        enc = LGBMCategoricalEncoder(use_native_categoricals=use_native_cat)
        enc.fit(X_fe)
        X_enc = enc.transform(X_fe)
        del X_fe; gc.collect()
        mlflow.log_param("use_native_categoricals", use_native_cat)
        mlflow.log_metric("n_category_cols", 
                          len(X_enc.select_dtypes('category').columns))

    with maybe_run(f"LightGBM_Feature_Selection_{run_prefix}"):
        sel = selector_class(**selector_params)
        sel.fit(X_enc, y)
        X_sel = sel.transform(X_enc)
        del X_enc; gc.collect()
        mlflow.log_param("selector", selector_class.__name__)
        mlflow.log_params(selector_params)
        mlflow.log_metric("features_before", 
                          X_sel.shape[1] + len(sel.dropped_cols_))
        mlflow.log_metric("features_after", X_sel.shape[1])
        mlflow.log_metric("features_dropped", len(sel.dropped_cols_))

    print(f"{run_prefix}: final shape {X_sel.shape}")
    return X_sel, cleaner, fe, enc, sel

In [20]:
import gc
X_sel_v1, cleaner_v1, fe_v1, enc_v1, sel_v1 = process_variation(
    X, y,
    version=1,
    null_threshold=0.80,
    use_native_cat=False,
    selector_class=VarianceSelector,
    selector_params={"threshold": 0.01},
    run_prefix="V1",
    log=False
)

V1: final shape (590540, 337)


In [15]:
X_sel_v2, cleaner_v2, fe_v2, enc_v2, sel_v2 = process_variation(
    X, y,
    version=2,
    null_threshold=0.50,
    use_native_cat=True,
    selector_class=CorrelationSelector,
    selector_params={"threshold": 0.95},
    run_prefix="V2"
)

🏃 View run LightGBM_Cleaning_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/7e01f767880244b496fdd7ca8962f366
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
🏃 View run LightGBM_Feature_Engineering_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/d5d54ca4911047068d87ab59a123f57a
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
🏃 View run LightGBM_Encoding_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/d86a3e3cf19c45788361aa04609e78ef
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
🏃 View run LightGBM_Feature_Selection_V2 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/d65d9a8d30e1405e93cf956d7150014e
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/ex

In [16]:
X_sel_v3, cleaner_v3, fe_v3, enc_v3, sel_v3 = process_variation(
    X, y,
    version=3,
    null_threshold=0.90,
    use_native_cat=True,
    selector_class=LGBMImportanceSelector,
    selector_params={"top_n": 150},
    run_prefix="V3"
)

🏃 View run LightGBM_Cleaning_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/a0ddcd21df5244489692ae19dfa5a68f
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
🏃 View run LightGBM_Feature_Engineering_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/075e325c5e254b19a275a779ae51090e
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
🏃 View run LightGBM_Encoding_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/21fef1ca10f04112a953e3cd37438273
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
🏃 View run LightGBM_Feature_Selection_V3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/f25b91111bed424abcf70ec3da857864
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/ex

In [24]:
SCALE_POS_WEIGHT = (y == 0).sum() / (y == 1).sum()
SKF = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

lgbm_param_sets = {
    "underfit": {
        "n_estimators": 50,
        "num_leaves": 8,
        "learning_rate": 0.3,
        "min_child_samples": 100,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "random_state": 42,
        "verbose": -1
    },
    "baseline": {
        "n_estimators": 300,
        "num_leaves": 31,
        "learning_rate": 0.05,
        "min_child_samples": 20,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "random_state": 42,
        "verbose": -1
    },
    "overfit": {
        "n_estimators": 500,
        "num_leaves": 256,
        "learning_rate": 0.05,
        "min_child_samples": 1,
        "reg_alpha": 0,
        "reg_lambda": 0,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "random_state": 42,
        "verbose": -1
    },
    "tuned": {
        "n_estimators": 500,
        "num_leaves": 64,
        "learning_rate": 0.05,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "random_state": 42,
        "verbose": -1
    }
}

In [18]:
def run_lgbm_cv(X_data, y_data, params, skf):
    train_aucs, val_aucs = [], []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
        X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
        y_tr, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        train_aucs.append(roc_auc_score(y_tr, model.predict_proba(X_tr)[:,1]))
        val_aucs.append(roc_auc_score(y_val, model.predict_proba(X_val)[:,1]))
        
        del X_tr, X_val; gc.collect()
    
    return np.mean(train_aucs), np.mean(val_aucs)

In [19]:
datasets = {"V1": X_sel_v1, "V2": X_sel_v2, "V3": X_sel_v3}

for var_name, X_data in datasets.items():
    for param_name, params in lgbm_param_sets.items():
        run_name = f"LightGBM_CV_{var_name}_{param_name}"
        print(f"Running {run_name}...")
        
        with mlflow.start_run(run_name=run_name):
            train_auc, val_auc = run_lgbm_cv(X_data, y, params, SKF)
            gap = train_auc - val_auc
            
            mlflow.log_params(params)
            mlflow.log_param("variation", var_name)
            mlflow.log_param("n_features", X_data.shape[1])
            mlflow.log_metric("train_auc", train_auc)
            mlflow.log_metric("val_auc", val_auc)
            mlflow.log_metric("overfit_gap", gap)
            
            print(f"  train={train_auc:.4f} | val={val_auc:.4f} | gap={gap:.4f}")

Running LightGBM_CV_V1_underfit...
  train=0.9014 | val=0.8928 | gap=0.0087
🏃 View run LightGBM_CV_V1_underfit at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/3b41e6fabe734e378fc93ef4e3e97c89
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
Running LightGBM_CV_V1_baseline...
  train=0.9592 | val=0.9380 | gap=0.0213
🏃 View run LightGBM_CV_V1_baseline at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/219f0130b3f44a6ab36e462ed50ce10a
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
Running LightGBM_CV_V1_overfit...
  train=1.0000 | val=0.9665 | gap=0.0335
🏃 View run LightGBM_CV_V1_overfit at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/e189b8f207034f9ca31a3863ea8b261f
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
Running LightGBM_CV_V

In [16]:
best_params = {
    "n_estimators": 500,
    "num_leaves": 64,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "random_state": 42,
    "verbose": -1
}

best_pipeline = Pipeline([
    ('cleaner', LGBMCleaningTransformer(null_threshold=0.80)),
    ('feature_engineer', LGBMFeatureEngineer(version=1)),
    ('encoder', LGBMCategoricalEncoder(use_native_categoricals=False)),
    ('selector', VarianceSelector(threshold=0.01)),
    ('model', LGBMClassifier(**best_params))
])

best_pipeline.fit(X, y)

Pipeline(steps=[('cleaner', LGBMCleaningTransformer()),
                ('feature_engineer', LGBMFeatureEngineer()),
                ('encoder', LGBMCategoricalEncoder()),
                ('selector', VarianceSelector()),
                ('model',
                 LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05,
                                n_estimators=500, num_leaves=64,
                                random_state=42, reg_alpha=0.1, reg_lambda=1.0,
                                scale_pos_weight=np.float64(27.579586700866283),
                                subsample=0.8, verbose=-1))])

In [17]:
with mlflow.start_run(run_name="LightGBM_Best_Pipeline"):

    mlflow.log_params(best_params)
    mlflow.log_param("null_threshold", 0.80)
    mlflow.log_param("fe_version", 1)
    mlflow.log_param("selector", "VarianceThreshold_0.01")
    mlflow.log_param("variation", "V1")
    mlflow.log_metric("val_auc", 0.9568)
    mlflow.log_metric("overfit_gap", 0.0338)
    
    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="lgbm_fraud_pipeline",
        registered_model_name="LightGBM_Fraud_Detection_Best"
    )

2026/05/08 03:12:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 03:12:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LightGBM_Fraud_Detection_Best'.
2026/05/08 03:13:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_Fraud_Detection_Best, version 1
Created version '1' of model 'LightGBM_Fraud_Detection_Best'.


🏃 View run LightGBM_Best_Pipeline at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/307574663f974288b110635044ba5c14
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1


In [ ]:
with mlflow.start_run(run_name="LightGBM_Best_Pipeline_v2"):
    mlflow.log_params(best_params)
    mlflow.log_metric("val_auc", 0.9568)
    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="lgbm_fraud_pipeline",
        registered_model_name="LightGBM_Fraud_Detection_Best"
    )

In [25]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

with mlflow.start_run(run_name="LightGBM_V1_stronger_regularization"):
    
    X_train_cv, X_val_cv, y_train_cv, y_val_cv = train_test_split(
        X_sel_v1, y, test_size=0.2, stratify=y, random_state=42
    )
    
    stronger_params = {
        "n_estimators": 2000,
        "num_leaves": 31,
        "learning_rate": 0.02,
        "min_child_samples": 50,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "reg_alpha": 1.0,
        "reg_lambda": 5.0,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "random_state": 42,
        "verbose": -1
    }
    
    model = LGBMClassifier(**stronger_params)
    model.fit(
        X_train_cv, y_train_cv,
        eval_set=[(X_val_cv, y_val_cv)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    train_auc = roc_auc_score(y_train_cv, model.predict_proba(X_train_cv)[:,1])
    val_auc = roc_auc_score(y_val_cv, model.predict_proba(X_val_cv)[:,1])
    gap = train_auc - val_auc
    
    mlflow.log_params(stronger_params)
    mlflow.log_param("early_stopping_rounds", 50)
    mlflow.log_param("best_iteration", model.best_iteration_)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", gap)
    
    print(f"Best iteration: {model.best_iteration_}")
    print(f"train={train_auc:.4f} | val={val_auc:.4f} | gap={gap:.4f}")
    print(f"Previous tuned gap was ~0.034 - did we reduce it?")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.14118
Best iteration: 3
train=0.8718 | val=0.8682 | gap=0.0036
Previous tuned gap was ~0.034 - did we reduce it?
🏃 View run LightGBM_V1_stronger_regularization at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/b4521c20db5e4ce5a0889976d0c7527e
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1


In [26]:
with mlflow.start_run(run_name="LightGBM_V1_balanced_early_stopping"):
    X_train_cv, X_val_cv, y_train_cv, y_val_cv = train_test_split(
        X_sel_v1, y, test_size=0.2, stratify=y, random_state=42
    )
    
    balanced_params = {
        "n_estimators": 2000,
        "num_leaves": 48,
        "learning_rate": 0.05,
        "min_child_samples": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.3,
        "reg_lambda": 2.0,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "random_state": 42,
        "verbose": -1
    }
    
    model = LGBMClassifier(**balanced_params)
    model.fit(
        X_train_cv, y_train_cv,
        eval_set=[(X_val_cv, y_val_cv)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    train_auc = roc_auc_score(y_train_cv, model.predict_proba(X_train_cv)[:,1])
    val_auc = roc_auc_score(y_val_cv, model.predict_proba(X_val_cv)[:,1])
    gap = train_auc - val_auc
    
    mlflow.log_params(balanced_params)
    mlflow.log_param("early_stopping_rounds", 50)
    mlflow.log_param("best_iteration", model.best_iteration_)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", gap)
    
    print(f"Best iteration: {model.best_iteration_}")
    print(f"train={train_auc:.4f} | val={val_auc:.4f} | gap={gap:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.137757
Best iteration: 1
train=0.8651 | val=0.8610 | gap=0.0042
🏃 View run LightGBM_V1_balanced_early_stopping at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/67d019d5cb2d42d280393e865ad941cc
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1


In [27]:
with mlflow.start_run(run_name="LightGBM_V1_early_stopping_auc"):
    X_train_cv, X_val_cv, y_train_cv, y_val_cv = train_test_split(
        X_sel_v1, y, test_size=0.2, stratify=y, random_state=42
    )
    
    balanced_params = {
        "n_estimators": 2000,
        "num_leaves": 48,
        "learning_rate": 0.05,
        "min_child_samples": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.3,
        "reg_lambda": 2.0,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "metric": "auc",
        "random_state": 42,
        "verbose": -1
    }
    
    model = LGBMClassifier(**balanced_params)
    model.fit(
        X_train_cv, y_train_cv,
        eval_set=[(X_val_cv, y_val_cv)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    
    train_auc = roc_auc_score(y_train_cv, model.predict_proba(X_train_cv)[:,1])
    val_auc = roc_auc_score(y_val_cv, model.predict_proba(X_val_cv)[:,1])
    gap = train_auc - val_auc
    
    mlflow.log_params(balanced_params)
    mlflow.log_param("early_stopping_rounds", 50)
    mlflow.log_param("best_iteration", model.best_iteration_)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", gap)
    
    print(f"Best iteration: {model.best_iteration_}")
    print(f"train={train_auc:.4f} | val={val_auc:.4f} | gap={gap:.4f}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.928614
[200]	valid_0's auc: 0.941904
[300]	valid_0's auc: 0.948251
[400]	valid_0's auc: 0.95278
[500]	valid_0's auc: 0.956246
[600]	valid_0's auc: 0.958474
[700]	valid_0's auc: 0.960302
[800]	valid_0's auc: 0.961852
[900]	valid_0's auc: 0.963039
[1000]	valid_0's auc: 0.964028
[1100]	valid_0's auc: 0.96487
[1200]	valid_0's auc: 0.965762
[1300]	valid_0's auc: 0.966321
[1400]	valid_0's auc: 0.966876
[1500]	valid_0's auc: 0.967426
[1600]	valid_0's auc: 0.968004
[1700]	valid_0's auc: 0.968327
[1800]	valid_0's auc: 0.968584
[1900]	valid_0's auc: 0.968847
[2000]	valid_0's auc: 0.969061
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.969061
Best iteration: 2000
train=0.9989 | val=0.9691 | gap=0.0298
🏃 View run LightGBM_V1_early_stopping_auc at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/2cc378cb7b6c484c98fc20e180085a12
🧪 View experiment at: https://d

In [28]:
best_params_v2 = {
    "n_estimators": 2000,
    "num_leaves": 48,
    "learning_rate": 0.05,
    "min_child_samples": 30,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.3,
    "reg_lambda": 2.0,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "metric": "auc",
    "random_state": 42,
    "verbose": -1
}

best_pipeline_v2 = Pipeline([
    ('cleaner', LGBMCleaningTransformer(null_threshold=0.80)),
    ('feature_engineer', LGBMFeatureEngineer(version=1)),
    ('encoder', LGBMCategoricalEncoder(use_native_categoricals=False)),
    ('selector', VarianceSelector(threshold=0.01)),
    ('model', LGBMClassifier(**best_params_v2))
])

best_pipeline_v2.fit(X, y)

with mlflow.start_run(run_name="LightGBM_Best_Pipeline_v3"):
    mlflow.log_params(best_params_v2)
    mlflow.log_param("early_stopping_used", True)
    mlflow.log_param("note", "improved regularization + early stopping")
    mlflow.log_metric("val_auc_single_split", 0.9691)
    mlflow.log_metric("overfit_gap", 0.0298)
    mlflow.sklearn.log_model(
        sk_model=best_pipeline_v2,
        artifact_path="lgbm_fraud_pipeline_v2",
        registered_model_name="LightGBM_Fraud_Detection_Best"
    )
    print("Improved pipeline saved to registry as version 3.")

2026/05/08 15:02:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 15:02:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LightGBM_Fraud_Detection_Best' already exists. Creating a new version of this model...
2026/05/08 15:02:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_Fraud_Detection_Best, version 3
Created version '3' of model 'LightGBM_Fraud_Detection_Best'.


Improved pipeline saved to registry as version 3.
🏃 View run LightGBM_Best_Pipeline_v3 at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/6922d9eed37746b6908ec1f50418898d
🧪 View experiment at: https://dagshub.com/amama22/ieee-cis-fraud-detection.mlflow/#/experiments/1
